# Lab 4 Workbook: From a Number to a Decision
## MPS311/439 Machine Learning

The same tasks as the [Lab 4 worksheet](https://ml.wxing.me/materials/lesson4/lab.html), with the code already in place: **nothing to copy**.

**What you do today.** The Python is given. What is left to you are the machine-learning decisions: the blanks `____` and the questions. That is the lab. A cell with a blank stops with an error until you fill it.

## How to work

1. **File → Save a copy in Drive**, and work in the copy so your answers are saved.
2. Run the cells from top to bottom (**▶** or **Shift + Enter**); each cell needs the cells above it.
3. Where you see **Your answer**, double-click the cell and write your answer **before** you run the next code cell. A guess written down first is how you find out what you really expected.
4. Fill each `____`, run the cell, and compare with the **Success check** below it.
5. Stuck on a blank? Open the folded Hint. The folded Answers are on the [worksheet](https://ml.wxing.me/materials/lesson4/lab.html); if you fall behind, copy one and carry on.
6. The **Try with AI** cells are for when you finish early, or after class.

# Part 0: What is the simplest guess worth? (10 minutes)

**Goal:** load the data, and find the accuracy that any classifier has to beat.

## 0.1 Load and split the data

Run the cell without changing it. The data come with scikit-learn, so nothing is downloaded.

In [ ]:
import numpy as np                                        # numbers and arrays
import matplotlib.pyplot as plt                           # plots
from sklearn.datasets import load_breast_cancer           # the 569 lumps
from sklearn.model_selection import train_test_split, KFold, cross_val_predict   # splitting and cross-validation
from sklearn.preprocessing import StandardScaler          # rescales columns
from sklearn.linear_model import LogisticRegression       # the model of today
from sklearn.metrics import confusion_matrix, recall_score, precision_score      # counting mistakes

data = load_breast_cancer()                               # 569 lumps, 30 measurements each
X = data.data                                             # the 30 measurements (features)
y = 1 - data.target                                       # sklearn uses 0 = malignant; we want 1 = malignant
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, same share of malignant in both parts
print(X_train.shape, X_test.shape)                        # how many lumps in each part
print(y_train.sum(), y_test.sum())                        # how many malignant in each part

**Success check:** `(426, 30) (143, 30)` and `159 53`.

## 0.2 The majority baseline

**Question first:** 159 of the 426 training lumps are malignant. If a "model" always guesses the more common class, what accuracy will it get? Write a guess, then fill the code.

**Your answer:** *(double-click to write)*

In [ ]:
share_benign = (y_train == ____).mean()                   # fraction of training lumps in that class
print("always guess benign:", share_benign)               # its accuracy

<details><summary><b>Hint</b> (only if you are stuck)</summary>

benign is labelled 0.

</details>

<details><summary><b>Success check</b> (open after you run the cell)</summary>

`always guess benign: 0.6267...`

</details>

**Questions:**

1. A model has accuracy 0.86 on these lumps. Is that "86% good"? Compare it with the baseline in one sentence.
2. In a disease that 1 person in 100 has, what accuracy does "always guess healthy" get? Give a number.

**Your answer:** *(double-click to write)*

# Part 1: Fit logistic regression (12 minutes)

**Goal:** fit the model on the 30 measurements and read its output as probabilities.

## 1.1 Fit on the raw measurements

Run the cell without changing it. It fits the model on the raw measurements.

In [ ]:
raw_model = LogisticRegression()                          # default settings: at most 100 steps
raw_model.fit(X_train, y_train)                           # fit on the raw measurements
print(X_train[:, 0].min(), X_train[:, 0].max())           # range of mean radius
print(X_train[:, 3].min(), X_train[:, 3].max())           # range of mean area

**Success check:** a yellow **ConvergenceWarning** appears, and the two ranges are about `7 to 28` and `144 to 2501`.

**Question:** the warning says the solver stopped before reaching the bottom. Using Lesson 3, what would you do to the columns first? One word.

Write your answer, then fill the two blanks below.

**Your answer:** *(double-click to write)*

In [ ]:
scaler = StandardScaler()                                 # mean and standard deviation per column
X_train_s = scaler.____(X_train)                          # learn them from the training lumps, and scale
X_test_s = scaler.____(X_test)                            # scale the test lumps the same way
print(X_train_s.mean(axis=0)[:3].round(2), X_train_s.std(axis=0)[:3].round(2))   # first three columns

<details><summary><b>Hint</b> (only if you are stuck)</summary>

as in Lab 3: one method learns and scales, the other only scales.

</details>

**Success check:** `[0. 0. 0.] [1. 1. 1.]` (the first number may print as `-0.`).

## 1.2 Probabilities, not classes

**Question first:** the model gives two numbers for each lump. What do you expect them to add up to?

**Your answer:** *(double-click to write)*

In [ ]:
model = LogisticRegression()                              # sigmoid model, trained with cross-entropy
model.fit(X_train_s, y_train)                             # fit on the standardised training lumps
prob_both = model.predict_proba(X_train_s[:5])                     # two probabilities for the first five training lumps
print(prob_both.round(3))                                 # column 0: benign, column 1: malignant
print(y_train[:5])                                        # the true labels of these five lumps

<details><summary><b>Success check</b> (open after you run the cell)</summary>

five rows of two numbers, each row adding up to 1; the labels are `[0 0 1 1 1]`. The first row is about `[0.944 0.056]`; the fourth is about `[0.253 0.747]`, a malignant lump the model is less sure about.

</details>

**Questions:**

1. Which column is the probability of malignant?
2. `model.predict` turns these into 0 or 1. Which threshold does it use?

**Your answer:** *(double-click to write)*

# Part 2: Count the two kinds of mistake (12 minutes)

**Goal:** use cross-validation inside the training set to see which mistakes the model makes, without touching the test set.

## 2.1 Cross-validated probabilities and the confusion matrix

Run the cell without changing it. As in Lesson 3, the training lumps are split into 5 folds; each lump gets a probability from a model that did not see it.

In [ ]:
kf = KFold(5, shuffle=True, random_state=0)               # 5 folds; shuffle first
prob_both = cross_val_predict(LogisticRegression(), X_train_s, y_train,
                              cv=kf, method="predict_proba")   # one probability pair per training lump
prob = prob_both[:, 1]                                    # column 1: probability of malignant
pred = (prob >= 0.5).astype(int)                          # threshold 0.5: 1 = malignant, 0 = benign
print(confusion_matrix(y_train, pred, labels=[1, 0]))     # rows: actual malignant, actual benign

**Success check:**

```
[[152   7]
 [  2 265]]
```

Read it like the lecture: first row, actually malignant (152 caught, **7 missed**); second row, actually benign (**2 false alarms**, 265 cleared).

## 2.2 Recall by hand

**Recall** is the share of malignant lumps that the model finds: caught divided by (caught plus missed). **Precision** is the share of lumps called malignant that really are malignant: caught divided by (caught plus false alarms). Fill the number of missed lumps from 2.1, then compare with scikit-learn.

In [ ]:
recall_by_hand = 152 / (152 + ____)                       # caught / (caught + missed)
print(round(recall_by_hand, 3))                           # your recall
print(round(recall_score(y_train, pred), 3))              # scikit-learn's recall
print(round(precision_score(y_train, pred), 3))           # precision: share of "malignant" calls that are right

<details><summary><b>Hint</b> (only if you are stuck)</summary>

the top-right number of the matrix.

</details>

**Success check:** `0.956`, `0.956` and `0.987`.

**Question:** the accuracy here is (152 + 265) / 426 = 0.979 (the two correct cells of your matrix over all 426 lumps), far above the baseline of 0.627. Still, 7 cancers are missed. In one sentence: why can accuracy hide this?

**Your answer:** *(double-click to write)*

**Try with AI (if you finish early, or after class)**

**You want:** to see how the cross-validated probabilities are spread for the two classes.

**Tell the AI:** `prob` holds one probability of malignant per training lump; `y_train` holds the true labels (1 = malignant, 0 = benign).

**Ask:** *"Draw two histograms of prob on the same axes, one for the lumps with y_train equal to 1 and one for y_train equal to 0, with 20 bins between 0 and 1 and a legend."*

**Check:** the two histograms contain `159` and `267` lumps; most malignant lumps sit near 1, most benign near 0.

# Part 3: Choose a threshold, then read the test set once (8 minutes)

**Goal:** see how the threshold trades missed cancers against false alarms, choose one for a clinic, and report it on the test set.

## 3.1 Try three thresholds

Run the cell without changing it. It compares the thresholds 0.5, 0.3 and 0.1.

In [ ]:
for t in [0.5, 0.3, 0.1]:                               # three thresholds to compare
    pred_t = (prob >= t).astype(int)                      # call malignant when probability >= t
    cm = confusion_matrix(y_train, pred_t, labels=[1, 0]) # the 2 x 2 table
    missed = cm[0, 1]                                     # actually malignant, called benign
    false_alarms = cm[1, 0]                               # actually benign, called malignant
    print("threshold", t, "missed", missed, "false alarms", false_alarms)

**Success check:**

```
threshold 0.5 missed 7 false alarms 2
threshold 0.3 missed 6 false alarms 9
threshold 0.1 missed 3 false alarms 27
```

**Question:** a missed cancer is the worst outcome for the clinic. Which threshold do you choose? Compare the number of missed cancers at 0.5 and at your threshold.

**Your answer:** *(double-click to write)*

## 3.2 The test set, once

The threshold was chosen inside the training set. Now, and only now, fit on all training lumps and read the test set once. Fill in your chosen threshold.

In [ ]:
model = LogisticRegression()                              # the same model
model.fit(X_train_s, y_train)                             # train on all 426 training lumps
prob_test = model.predict_proba(X_test_s)[:, 1]           # probability of malignant for each test lump
pred_test = (prob_test >= ____).astype(int)               # your chosen threshold
print(confusion_matrix(y_test, pred_test, labels=[1, 0])) # the test set, read once

<details><summary><b>Hint</b> (only if you are stuck)</summary>

the threshold you chose in 3.1.

</details>

**Success check (threshold 0.1):**

```
[[53  0]
 [11 79]]
```

All 53 malignant test lumps are found; 11 of the 90 benign lumps are false alarms. If you chose 0.3, you get `[[51 2] [6 84]]`: 2 missed, 6 false alarms.

**Try with AI (if you finish early, or after class)**

**You want:** to see the whole trade-off, not only three thresholds.

**Tell the AI:** `prob` holds the cross-validated probability of malignant for each training lump; `y_train` holds the labels (1 = malignant).

**Ask:** *"For thresholds from 0.05 to 0.95 in steps of 0.05, count the missed malignant lumps (y_train is 1 but prob is below the threshold) and the false alarms (y_train is 0 but prob is at least the threshold). Plot both counts against the threshold on one figure."*

**Check:** at threshold 0.5 the two counts are `7` and `2`; missed cancers go down and false alarms go up as the threshold falls.

**Try with AI (if you finish early, or after class)**

**You want:** to see what a multi-class logistic regression looks at, as in Assignment 2.

Run this cell first, without changing it. It downloads the course's MNIST subset (10,000 handwritten digits of 28 × 28 pixels, about 1.6 MB; Labs 5 to 8 use it too) and fits logistic regression. It takes about half a minute.

In [ ]:
import urllib.request                                     # to download a file
DATA_URL = "https://raw.githubusercontent.com/wayXing/mps311-439-course-materials/main/lecture-slids/lessons/data/mnist10k.npz"
urllib.request.urlretrieve(DATA_URL, "mnist10k.npz")      # 10,000 MNIST images, about 1.6 MB
mnist = np.load("mnist10k.npz")                           # the file holds two arrays, X and y
X_digits = mnist["X"] / 255                               # 784 pixels per image (28 x 28), scaled to 0 to 1
y_digits = mnist["y"]                                     # the digit, 0 to 9
X_digits_train, X_digits_test, y_digits_train, y_digits_test = train_test_split(
    X_digits, y_digits, test_size=0.25, random_state=0, stratify=y_digits)   # 7500 to train, 2500 to test
digit_model = LogisticRegression(max_iter=1000, C=0.1)    # more steps; C = 0.1 regularises (Lesson 3), chosen by cross-validation
digit_model.fit(X_digits_train, y_digits_train)           # softmax and cross-entropy happen inside
print(digit_model.score(X_digits_test, y_digits_test))    # accuracy on the test images

**Tell the AI:** `digit_model.coef_` has 10 rows (one per digit) of 784 weights, one per pixel of a 28 × 28 image.

**Ask:** *"Show the 784 weights of each of the 10 rows of digit_model.coef_ as a 28 × 28 image, in a grid of 2 rows and 5 columns, with a red-blue colour map centred at zero (positive red, negative blue) and the digit as each title."*

**Check:** the printed accuracy is about `0.908`, and there are 10 small images; the one for digit 0 has a blue centre and a red ring around it.

# After class

How today's lab helps with **Assignment 2**, the reflection questions and the **Advanced Study** are at the end of the [worksheet](https://ml.wxing.me/materials/lesson4/lab.html).